# 08 Current collector (IA-COL-001)

**Requirement:** `IA-COL-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

## Governing equations

$$V_\mathrm{ohm} = I\,R(T,\mathrm{degradation})$$

No fake lag on resistance.
Anode branch-plus-primary, ORR dual-face tabs, Ni-coated-steel + EPDM (US FIGS. 4B-4C),
SS mesh in iron ribs (EP), can-negative housing (US Claim 5).
Ohmic drop in $\mathrm{V}$ for current in $\mathrm{A}$; $R$ in $\mathrm{\Omega}$.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path

setup_path()
from plant_sim.params import CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G, default_params

In [ ]:
from IPython import get_ipython

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 18,
        "figure.figsize": (11, 8.5),
        "figure.dpi": 300,
        "figure.facecolor": "w",
        "figure.edgecolor": "k",
    }
)
P = default_params()
print("KOH default", P.c_KOH_mol_m3 / 1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G, 2), round(CAPACITY_MAGNETITE_MAH_G, 2))

## Simulation setup

Integrate the model once. Plot sections below each document what is shown, why it matters for patent/requirement verification, and the equations that govern that view.


In [ ]:
from plant_sim.components.current_collector import CurrentCollector
from plant_sim.components.base import integrate_component

col = CurrentCollector(P)
t, y = integrate_component(
    lambda t, y: col.rhs(t, y, {"I_cell_A": 8.0, "T_amb_K": P.T_ep_sim_K}, {}), col.y0(), (0, 1500), n_eval=100
)
V = [col.outputs(float(ti), y[:, i], {"I_cell_A": 8.0}, {})["V_ohm_V"] for i, ti in enumerate(t)]

## Current-collector ohmic drop

### What this plot illustrates

Ohmic voltage loss on the collector as current and degradation evolve: $V_\mathrm{ohm}=I R(T,\mathrm{deg})$.

### Governing equations

$$V_\mathrm{ohm} = I\,R(T,\mathrm{degradation})$$

$R$ rises weakly with $T$ and with accumulated degradation state.


In [ ]:
plot_frame = pd.DataFrame(
    {
        "t": (t / 60),
        "T": (y[0]),
        "deg": (y[1]),
        "Vohm": (V),
    }
)
plot_long = plot_frame.melt(id_vars="t", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="t", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("t (min)")
ax.set_ylabel("T / deg / V")
ax.set_title("Collector ohmic loss")
ax.text(0.0, -0.22, "IA-COL-001", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)

## Verification against patents / SSS002

In [ ]:
print("V_ohm = I * R(T, deg) with no extra lag state.")